In [ ]:
from statsmodels.stats.inter_rater import fleiss_kappa
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

In [ ]:
metrics = [
    'relevance',
    'clarity',
    'answerability',
    'challenging',
    'value',
    'language',
    'bloom_rating',
    "blooms_level_score"
 ]
metric_labels = {
    'relevance': 'Relevance',
    'clarity': 'Clarity',
    'answerability': 'Answerability',
    'challenging': 'Challenging',
    'value': 'Value',
    'language': 'Language',
    'bloom_rating': "Bloom's Level (Student)",
    'blooms_level_score': "Bloom's Level (Score)"
}

In [ ]:
BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

qualitative_base_path = os.path.join(
    BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp2"
  )

student_eval_files = {
    1: os.path.join(qualitative_base_path, "exp2_eval_s1.csv"),
    2: os.path.join(qualitative_base_path, "exp2_eval_s2.csv"),
    3: os.path.join(qualitative_base_path, "exp2_eval_s3.csv"),
}

output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp2/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

tables = {}
plots = {}

LABEL_MAPPING = {
    'anthropic': 'Anthropic',
    'openai': 'OpenAI',
    'deepseek': 'DeepSeek',
    'xai': 'xAI',
    'google': 'Google',
    'mcq': 'MCQ',
    'open_ended': 'Open-Ended',
    'exp2a': 'Type Only',
    'exp2b': 'Bloom Only',
    'exp2c': 'Type + Bloom',
}

print("Setup completed successfully")
print(f"Data source folder: {qualitative_base_path}")
print(f"Student eval files: {student_eval_files}")
print(f"Output tables: {output_tables_path}")
print(f"Output plots: {output_plots_path}")

In [ ]:
def load_student_data(student_id):
    if student_id not in student_eval_files:
        raise KeyError(f"Unknown student_id={student_id}. Known: {list(student_eval_files.keys())}")
    csv_path = student_eval_files[student_id]
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Student {student_id} data not found at {csv_path}")
    df = None
    for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    if df is None:
        raise UnicodeDecodeError("utf-8", b"", 0, 1, f"Could not read {csv_path} with any encoding")
    
    for col in ['comments', 'answer_problems']:
        if col in df.columns:
            df = df.drop(columns=[col])

    if 'sample_id' in df.columns:
        df['sample_id'] = df['sample_id'].astype(str)
        df['question_num'] = df['sample_id'].str.extract('(\\d+)').astype(int).iloc[:, 0]
    else:
        df['question_num'] = range(1, len(df) + 1)

    if 'experiment' not in df.columns:
        df['experiment'] = 'exp2'

    df['student'] = student_id

    print(f"Loaded student_{student_id}: {len(df)} rows from {os.path.basename(csv_path)}")
    return {'exp2': df}

In [ ]:
all_data = {}
for i in range(1, 4):
    all_data[f"student_{i}"] = load_student_data(i)

In [ ]:
def combine_data():
    combined = []
    print("\n--- Combining all available student data ---")
    for student, experiments in all_data.items():
        print(f"\n{student}:")
        for exp, df in experiments.items():
            if df is None or df.empty:
                continue
            print(f"  {exp}: {len(df)} evaluations")
            combined.append(df)
    if combined:
        return pd.concat(combined, ignore_index=True)
    return pd.DataFrame()

df_combined = combine_data()
print(f"\n=== FINAL SUMMARY ===")
print(f"Total evaluations: {len(df_combined)}")
print(f"Columns: {list(df_combined.columns)}")

sampled_meta_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/sampled/exp2_sampled.csv")
if os.path.exists(sampled_meta_path):
    df_sampled = pd.read_csv(sampled_meta_path)
    df_sampled = df_sampled.copy()
    df_sampled['sample_id_int'] = range(1, len(df_sampled) + 1)
    df_sampled['sample_id'] = df_sampled['sample_id_int'].apply(lambda x: f"{int(x):03d}")
    df_sampled = df_sampled[['sample_id', 'llm', 'question_type', 'bloom_idx']]
    df_sampled['sample_id'] = df_sampled['sample_id'].astype(str)
    if 'sample_id' in df_combined.columns:
        df_combined['sample_id'] = df_combined['sample_id'].astype(str).str.extract(r'(\d+)').iloc[:, 0].astype(int).apply(lambda x: f"{int(x):03d}")
        before_cols = set(df_combined.columns)
        df_combined = df_combined.merge(df_sampled, on='sample_id', how='left', suffixes=('', '_meta'))
        if 'question_type_meta' in df_combined.columns:
            df_combined['question_type'] = df_combined['question_type_meta'].combine_first(df_combined.get('question_type'))
            df_combined = df_combined.drop(columns=['question_type_meta'])
        added_cols = [c for c in df_combined.columns if c not in before_cols]
        print(f"Joined exp2_sampled metadata. Added columns: {added_cols}")
        print("LLM counts (incl NaN):")
        print(df_combined['llm'].value_counts(dropna=False))
    else:
        print("WARNING: df_combined has no 'sample_id' column — cannot join exp2_sampled metadata.")
else:
    print(f"WARNING: exp2 sampled metadata not found at: {sampled_meta_path}")

# Experiment 2: Descriptive Statistics

Analysis of prompt engineering approaches for question generation.

In [ ]:
def convert_bloom_rating_simple(value, experiment, given_level=None):
    bloom_map = {
        6: 10,
        5: 8.5,
        4: 7,
        3: 4.5,
        2: 3,
        1: 1.5
    }

    def extract_levels(val):
        if isinstance(val, str):
            nums = re.findall(r'\d+', val)
            return [int(n) for n in nums] if nums else []
        elif isinstance(val, (int, float)) and not pd.isnull(val):
            return [int(val)]
        return []

    rater_levels = extract_levels(value)
    if not rater_levels:
        return np.nan

    if given_level is None or (isinstance(given_level, float) and pd.isnull(given_level)):
        max_level = max(rater_levels)
        return bloom_map.get(max_level, np.nan)

    target_levels = extract_levels(given_level)
    if not target_levels:
        return np.nan

    best_score = 0
    for r in rater_levels:
        for g in target_levels:
            if r == g:
                best_score = max(best_score, 10)
            elif abs(r - g) == 1:
                best_score = max(best_score, 5)
    return best_score

In [ ]:
print("EXPERIMENT 2 - DESCRIPTIVE STATISTICS (ALL RATINGS)")
print("="*70)
print(f"Number of ratings in df_combined: {len(df_combined)}")

df_numeric = df_combined.copy()

print("Converting metrics to numeric (all ratings)...")
for metric in metrics:
    if metric == 'bloom_rating':
        df_numeric[metric] = df_combined[metric]
    elif metric == 'blooms_level_score':
        def bloom_score_apply(row):
            target = None
            if 'bloom_idx' in row and not pd.isna(row['bloom_idx']):
                target = row['bloom_idx']
            if row.get('experiment') in ['exp2b', 'exp2c'] and 'bloom_original' in row and not pd.isnull(row['bloom_original']):
                target = row['bloom_original']
            return convert_bloom_rating_simple(row.get('bloom_rating'), row.get('experiment', 'exp2'), target)
        df_numeric[metric] = df_combined.apply(bloom_score_apply, axis=1)
    else:
        if metric in df_combined.columns:
            df_numeric[metric] = pd.to_numeric(df_combined[metric], errors='coerce')
        else:
            df_numeric[metric] = np.nan

display(df_numeric)

In [ ]:
desired_metrics = [
    'relevance',
    'clarity',
    'answerability',
    'challenging',
    'value',
    'language',
    'blooms_level_score',
    'total_score',
 ]

if 'blooms_level_score' not in df_numeric.columns:
    df_numeric['blooms_level_score'] = np.nan

component_metrics = [m for m in desired_metrics if m not in ('total_score',) and m in df_numeric.columns]
df_numeric['total_score'] = df_numeric[component_metrics].sum(axis=1, min_count=1)
numeric_metrics = desired_metrics.copy()
metric_labels['total_score'] = 'Total Score'

if len(df_numeric) > 0 and numeric_metrics:
    print("\nOverall Statistics (selected metrics):")
    overall_stats = df_numeric[numeric_metrics].describe().round(2)
    display(overall_stats)

    print("\nStatistics by Experiment:")
    exp_stats = df_numeric.groupby('experiment')[numeric_metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    display(exp_stats)
else:
    print("\nNo ratings available for statistics.")
    overall_stats = pd.DataFrame()
    exp_stats = pd.DataFrame()

In [ ]:
print("\n=== VISUALIZATION: PERFORMANCE BY QUESTION TYPE (4x2) ===")
if len(df_numeric) > 0 and numeric_metrics:
    fig, axes = plt.subplots(4, 2, figsize=(14, 20))
    axes = axes.flatten()

    if 'question_type' not in df_numeric.columns:
        print("No 'question_type' column found — cannot plot by question type.")
    else:
        qt_order = ['mcq', 'open_ended']
        present_qt = [q for q in qt_order if q in set(df_numeric['question_type'].dropna().unique())]
        if present_qt:
            qt_order = present_qt
        else:
            qt_order = sorted(df_numeric['question_type'].dropna().unique())

        for i, metric in enumerate(numeric_metrics):
            df_plot = df_numeric[[metric, 'question_type']].dropna()
            metric_display = metric_labels.get(metric, metric)

            if len(df_plot) > 0:
                colors = sns.color_palette("Set2", n_colors=len(qt_order))
                sns.boxplot(
                    data=df_plot,
                    x='question_type',
                    y=metric,
                    ax=axes[i],
                    palette=colors,
                    order=qt_order,
                    medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'},
                )
                for j, qt in enumerate(qt_order):
                    qt_data = df_plot[df_plot['question_type'] == qt][metric]
                    if len(qt_data) > 0:
                        mean_val = qt_data.mean()
                        axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, edgecolor='darkred', linewidth=1)

                if metric == 'total_score':
                    axes[i].set_ylim(0, 70)
                else:
                    axes[i].set_ylim(0, 10)

                axes[i].set_title(f'{metric_display}', fontsize=12, fontweight='bold')
                axes[i].set_xlabel('Question Type')
                axes[i].set_ylabel(metric_display)
                axes[i].grid(True, alpha=0.3)
                axes[i].set_xticklabels([LABEL_MAPPING.get(v, v) for v in qt_order], rotation=0)
            else:
                axes[i].set_title(f'{metric_display} - No Data')
                axes[i].text(0.5, 0.5, 'No Data', ha='center', va='center', transform=axes[i].transAxes)

        plt.suptitle('Experiment 2: Performance by Question Type', fontsize=16, fontweight='bold', y=0.995)
        plt.subplots_adjust(top=0.92)
        plt.tight_layout()
        if 'output_plots_path' in locals():
            figpath = os.path.join(output_plots_path, "exp2_boxplots_all_metrics_by_question_type.png")
            fig.savefig(figpath)
            print(f"Saved question-type boxplot grid: {figpath}")
        plt.show()
        print("Visualization by question type complete.")
else:
    print("No data available for visualization.")

# Metric Analysis by LLM

The following section provides boxplots and summary tables for all metrics, grouped by LLM (language model).

In [ ]:
print("\n=== METRIC ANALYSIS BY LLM ===")

if 'df_numeric' in locals() and len(df_numeric) > 0 and numeric_metrics:
    if 'llm' not in df_numeric.columns:
        print("No 'llm' column found in df_numeric yet — skipping LLM-based analysis.")
        print("Once you add/join LLM metadata (llm per sample_id), this section will run automatically.")
    else:
        fig, axes = plt.subplots(4, 2, figsize=(14, 20))
        axes = axes.flatten()

        llm_order = sorted(df_numeric['llm'].dropna().unique())
        for i, metric in enumerate(numeric_metrics):
            df_plot = df_numeric[[metric, 'llm']].dropna()
            metric_display = metric_labels.get(metric, metric)
            if len(df_plot) > 0:
                colors = sns.color_palette("Set2", n_colors=len(llm_order))
                sns.boxplot(
                    data=df_plot,
                    x='llm',
                    y=metric,
                    ax=axes[i],
                    palette=colors,
                    order=llm_order,
                    medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'},
                )
                for j, llm_name in enumerate(llm_order):
                    llm_data = df_plot[df_plot['llm'] == llm_name][metric]
                    if len(llm_data) > 0:
                        mean_val = llm_data.mean()
                        axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, edgecolor='darkred', linewidth=1)
                if metric == 'total_score':
                    axes[i].set_ylim(0, 70)
                else:
                    axes[i].set_ylim(0, 10)
                axes[i].set_title(f'{metric_display} by LLM', fontsize=12, fontweight='bold')
                axes[i].set_xlabel('LLM')
                axes[i].set_ylabel(metric_display)
                axes[i].grid(True, alpha=0.3)
                axes[i].set_xticklabels([LABEL_MAPPING.get(l, str(l).title()) for l in llm_order], rotation=0)
            else:
                axes[i].set_title(f'{metric_display} - No Data')
                axes[i].text(0.5, 0.5, 'No Data', ha='center', va='center', transform=axes[i].transAxes)

        plt.suptitle('Experiment 2: LLM Performance across all Criteria', fontsize=16, fontweight='bold', y=0.995)
        plt.subplots_adjust(top=0.92)
        plt.tight_layout()
        if 'output_plots_path' in locals():
            figpath = os.path.join(output_plots_path, "exp2_boxplots_all_metrics_by_llm.png")
            fig.savefig(figpath)
            print(f"Saved collective LLM boxplot: {figpath}")
        plt.show()
        print("Visualization by LLM complete.")

        print("\nSummary Table: Metrics by LLM (mean, std, count)")
        llm_stats = df_numeric.groupby('llm')[numeric_metrics].agg(['mean', 'std', 'median', 'count']).round(2)
        tables['exp2_metrics_by_llm'] = llm_stats
        display(llm_stats)
        if 'output_tables_path' in locals():
            llm_stats.to_csv(os.path.join(output_tables_path, "exp2_metrics_by_llm.csv"))
            print(f"Saved metrics by LLM table to {output_tables_path}")

In [ ]:
print("\n=== HEATMAP: VALUE by LLM vs Question Type (mean ± std) ===")

if 'df_numeric' in locals() and len(df_numeric) > 0:
    df_hm = df_numeric[['llm', 'question_type', 'value']].dropna()
    if df_hm.empty:
        print("No non-null 'value' ratings available yet.")
    else:
        mean_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='value', aggfunc='mean')
        std_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='value', aggfunc='std')
        count_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='value', aggfunc='count')
        tables['exp2_value_llm_x_question_type_mean'] = mean_tbl.round(3)
        tables['exp2_value_llm_x_question_type_std'] = std_tbl.round(3)
        tables['exp2_value_llm_x_question_type_count'] = count_tbl.astype(int)
        display(mean_tbl.round(2))

        annot = mean_tbl.copy()
        for r in range(mean_tbl.shape[0]):
            for c in range(mean_tbl.shape[1]):
                m = mean_tbl.iloc[r, c]
                s = std_tbl.iloc[r, c]
                n = count_tbl.iloc[r, c]
                if pd.notna(m):
                    if pd.notna(s):
                        annot.iloc[r, c] = f"{m:.1f}\n({s:.1f})\n n={int(n)}"
                    else:
                        annot.iloc[r, c] = f"{m:.1f}\n n={int(n)}"
                else:
                    annot.iloc[r, c] = ""

        fig, ax = plt.subplots(1, 1, figsize=(7, 10))
        plots['exp2_value_heatmap_llm_x_question_type'] = fig
        llm_labels = [LABEL_MAPPING.get(l, str(l).title()) for l in mean_tbl.index]
        qt_labels = [LABEL_MAPPING.get(q, str(q)) for q in mean_tbl.columns]
        sns.heatmap(
            mean_tbl,
            annot=annot,
            fmt='',
            cmap='RdYlBu_r',
            vmin=0,
            vmax=10,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Value (Mean)'},
            yticklabels=llm_labels,
            xticklabels=qt_labels,
            ax=ax,
        )
        ax.set_title("Value by LLM vs Question Type\nValues: Mean (Std) and n", fontsize=13, fontweight='bold', pad=14)
        ax.set_xlabel("Question Type")
        ax.set_ylabel("LLM")
        plt.tight_layout()
        if 'output_plots_path' in locals():
            figpath = os.path.join(output_plots_path, "exp2_value_heatmap_llm_x_question_type.png")
            fig.savefig(figpath, dpi=300, bbox_inches='tight')
            print(f"Saved heatmap: {figpath}")
        plt.show()

        if 'output_tables_path' in locals():
            mean_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_value_llm_x_question_type_mean.csv"))
            std_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_value_llm_x_question_type_std.csv"))
            count_tbl.astype(int).to_csv(os.path.join(output_tables_path, "exp2_value_llm_x_question_type_count.csv"))
            print(f"Saved value heatmap tables to {output_tables_path}")

In [ ]:
print("\n=== HEATMAP: Total Score by LLM vs Question Type (mean ± std) ===")

if 'df_numeric' in locals() and len(df_numeric) > 0:
    # total_score is created in the earlier stats cell; guard in case cells were run out of order
    if 'total_score' not in df_numeric.columns:
        print("NOTE: 'total_score' not in df_numeric yet. Run the descriptive stats cell that computes total_score first.")
    else:
        df_hm = df_numeric[['llm', 'question_type', 'total_score']].dropna()
        if df_hm.empty:
            print("No non-null 'total_score' values available yet.")
        else:
            mean_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='total_score', aggfunc='mean')
            std_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='total_score', aggfunc='std')
            count_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='total_score', aggfunc='count')

            tables['exp2_total_score_llm_x_question_type_mean'] = mean_tbl.round(3)
            tables['exp2_total_score_llm_x_question_type_std'] = std_tbl.round(3)
            tables['exp2_total_score_llm_x_question_type_count'] = count_tbl.astype(int)
            display(mean_tbl.round(2))

            annot = mean_tbl.copy()
            for r in range(mean_tbl.shape[0]):
                for c in range(mean_tbl.shape[1]):
                    m = mean_tbl.iloc[r, c]
                    s = std_tbl.iloc[r, c]
                    n = count_tbl.iloc[r, c]
                    if pd.notna(m):
                        if pd.notna(s):
                            annot.iloc[r, c] = f"{m:.1f}\n({s:.1f})\n n={int(n)}"
                        else:
                            annot.iloc[r, c] = f"{m:.1f}\n n={int(n)}"
                    else:
                        annot.iloc[r, c] = ""

            fig, ax = plt.subplots(1, 1, figsize=(7, 10))
            plots['exp2_total_score_heatmap_llm_x_question_type'] = fig
            llm_labels = [LABEL_MAPPING.get(l, str(l).title()) for l in mean_tbl.index]
            qt_labels = [LABEL_MAPPING.get(q, str(q)) for q in mean_tbl.columns]
            sns.heatmap(
                mean_tbl,
                annot=annot,
                fmt='',
                cmap='RdYlBu_r',
                vmin=0,
                vmax=70,
                linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': 'Total Score (Mean)'},
                yticklabels=llm_labels,
                xticklabels=qt_labels,
                ax=ax,
            )
            ax.set_title("Total Score by LLM vs Question Type\nValues: Mean (Std) and n", fontsize=13, fontweight='bold', pad=14)
            ax.set_xlabel("Question Type")
            ax.set_ylabel("LLM")
            plt.tight_layout()
            if 'output_plots_path' in locals():
                figpath = os.path.join(output_plots_path, "exp2_total_score_heatmap_llm_x_question_type.png")
                fig.savefig(figpath, dpi=300, bbox_inches='tight')
                print(f"Saved heatmap: {figpath}")
            plt.show()

            if 'output_tables_path' in locals():
                mean_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_total_score_llm_x_question_type_mean.csv"))
                std_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_total_score_llm_x_question_type_std.csv"))
                count_tbl.astype(int).to_csv(os.path.join(output_tables_path, "exp2_total_score_llm_x_question_type_count.csv"))
                print(f"Saved total_score heatmap tables to {output_tables_path}")

print("\n=== HEATMAP: Bloom Score (blooms_level_score) by LLM vs Question Type (mean ± std) ===")

if 'df_numeric' in locals() and len(df_numeric) > 0:
    if 'blooms_level_score' not in df_numeric.columns:
        print("NOTE: 'blooms_level_score' not in df_numeric yet. Run the Bloom conversion cell first.")
    else:
        df_hm = df_numeric[['llm', 'question_type', 'blooms_level_score']].dropna()
        if df_hm.empty:
            print("No non-null Bloom score values available yet.")
        else:
            mean_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='blooms_level_score', aggfunc='mean')
            std_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='blooms_level_score', aggfunc='std')
            count_tbl = df_hm.pivot_table(index='llm', columns='question_type', values='blooms_level_score', aggfunc='count')

            tables['exp2_bloom_score_llm_x_question_type_mean'] = mean_tbl.round(3)
            tables['exp2_bloom_score_llm_x_question_type_std'] = std_tbl.round(3)
            tables['exp2_bloom_score_llm_x_question_type_count'] = count_tbl.astype(int)
            display(mean_tbl.round(2))

            annot = mean_tbl.copy()
            for r in range(mean_tbl.shape[0]):
                for c in range(mean_tbl.shape[1]):
                    m = mean_tbl.iloc[r, c]
                    s = std_tbl.iloc[r, c]
                    n = count_tbl.iloc[r, c]
                    if pd.notna(m):
                        if pd.notna(s):
                            annot.iloc[r, c] = f"{m:.1f}\n({s:.1f})\n n={int(n)}"
                        else:
                            annot.iloc[r, c] = f"{m:.1f}\n n={int(n)}"
                    else:
                        annot.iloc[r, c] = ""

            fig, ax = plt.subplots(1, 1, figsize=(7, 10))
            plots['exp2_bloom_score_heatmap_llm_x_question_type'] = fig
            llm_labels = [LABEL_MAPPING.get(l, str(l).title()) for l in mean_tbl.index]
            qt_labels = [LABEL_MAPPING.get(q, str(q)) for q in mean_tbl.columns]
            sns.heatmap(
                mean_tbl,
                annot=annot,
                fmt='',
                cmap='RdYlBu_r',
                vmin=0,
                vmax=10,
                linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': "Bloom score (Mean)"},
                yticklabels=llm_labels,
                xticklabels=qt_labels,
                ax=ax,
            )
            ax.set_title("Bloom score by LLM vs Question Type\nValues: Mean (Std) and n", fontsize=13, fontweight='bold', pad=14)
            ax.set_xlabel("Question Type")
            ax.set_ylabel("LLM")
            plt.tight_layout()
            if 'output_plots_path' in locals():
                figpath = os.path.join(output_plots_path, "exp2_bloom_score_heatmap_llm_x_question_type.png")
                fig.savefig(figpath, dpi=300, bbox_inches='tight')
                print(f"Saved heatmap: {figpath}")
            plt.show()

            if 'output_tables_path' in locals():
                mean_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_bloom_score_llm_x_question_type_mean.csv"))
                std_tbl.round(3).to_csv(os.path.join(output_tables_path, "exp2_bloom_score_llm_x_question_type_std.csv"))
                count_tbl.astype(int).to_csv(os.path.join(output_tables_path, "exp2_bloom_score_llm_x_question_type_count.csv"))
                print(f"Saved bloom score heatmap tables to {output_tables_path}")

# Inter-Student Reliability Analysis

Analysis of agreement between student evaluators on the quality of generated questions.

In [ ]:
print("\n" + "="*60)
print("FLEISS' KAPPA INTER-RATER RELIABILITY ANALYSIS")
print("="*60)


def make_global_question_id(row):
    if row['experiment'] == 'exp2a':
        return f"{row['experiment']}|{row['llm']}|{row['question_id']}|{row['question_type']}"
    elif row['experiment'] == 'exp2b':
        return f"{row['experiment']}|{row['llm']}|{row['bloom_original']}"
    elif row['experiment'] == 'exp2c':
        return f"{row['experiment']}|{row['llm']}|{row['bloom_original']}|{row['question_type']}"
    else:
        return None

df_numeric['global_question_id'] = df_numeric.apply(make_global_question_id, axis=1)

In [ ]:
def extract_max_bloom_rating(val):
    if isinstance(val, str):
        nums = re.findall(r'\d+', val)
        return max([int(n) for n in nums]) if nums else np.nan
    elif isinstance(val, (int, float)) and not pd.isnull(val):
        return int(val)
    return np.nan

In [ ]:
def kappa_level(k):
    if k < 0:
        return "Invalid"
    if k < 0.2:
        return "Slight"
    elif k < 0.4:
        return "Fair"
    elif k < 0.6:
        return "Moderate"
    elif k < 0.8:
        return "Substantial"
    else:
        return "Almost Perfect"

In [ ]:
def calculate_fleiss(df, metrics):
    results = []
    df = df.copy()

    # Handle all needed 0-10 metrics
    for metric in metrics:
        if metric in ('bloom_rating', 'blooms_level_score', 'total_score'):
            continue
        pivot = df.pivot_table(index='global_question_id', columns='student', values=metric)
        pivot = pivot.dropna(axis=0)
        ratings_matrix = pivot.applymap(lambda x: max(1, min(10, int(round(x)))) if pd.notnull(x) else np.nan).values
        fleiss_table = np.zeros((ratings_matrix.shape[0], 10))
        for i, row in enumerate(ratings_matrix):
            for r in row:
                if not np.isnan(r) and 1 <= r <= 10:
                    fleiss_table[i, int(r)-1] += 1
        all_identical = np.all((fleiss_table == np.max(fleiss_table, axis=1, keepdims=True)) | (fleiss_table == 0), axis=1)
        try:
            if np.all(all_identical):
                kappa = 1.0
            else:
                kappa = fleiss_kappa(fleiss_table)
        except Exception:
            kappa = np.nan


        flat_valid = [r for row in ratings_matrix for r in row if not np.isnan(r)]
        mean_rating = float(np.mean(flat_valid)) if flat_valid else np.nan
        std_rating = float(np.std(flat_valid)) if flat_valid else np.nan

        results.append({
            'Metric': metric_labels.get(metric, metric),
            'Fleiss_Kappa': kappa,
            'Agreement_Level': kappa_level(kappa) if not np.isnan(kappa) else 'No data',
            'N_Questions': int(ratings_matrix.shape[0]),
            'N_Raters': int(pivot.shape[1]) if pivot.shape[0] > 0 else 0,
            'Mean_Rating': round(mean_rating, 2) if not np.isnan(mean_rating) else np.nan,
            'Std_Rating': round(std_rating, 2) if not np.isnan(std_rating) else np.nan
        })

    df['bloom_rating_numeric'] = df['bloom_rating'].apply(extract_max_bloom_rating)
    pivot = df.pivot_table(index='global_question_id', columns='student', values='bloom_rating_numeric')
    pivot = pivot.dropna(axis=0)
    ratings_matrix = pivot.values
    ratings_matrix = np.array([[max(1, min(6, int(r))) for r in row] for row in ratings_matrix if not any(pd.isnull(row))])

    fleiss_table = np.zeros((len(ratings_matrix), 6))
    for i, row in enumerate(ratings_matrix):
        for r in row:
            if 1 <= r <= 6:
                fleiss_table[i, int(r) - 1] += 1

    all_identical = np.all(
        (fleiss_table == np.max(fleiss_table, axis=1, keepdims=True)) | (fleiss_table == 0),
        axis=1
    )
    try:
        if np.all(all_identical):
            kappa = 1.0
        else:
            kappa = fleiss_kappa(fleiss_table)
    except Exception:
        kappa = np.nan

    flat_valid = [r for row in ratings_matrix for r in row]
    mean_rating = float(np.mean(flat_valid)) if flat_valid else np.nan
    std_rating = float(np.std(flat_valid)) if flat_valid else np.nan

    results.append({
        'Metric': 'Bloom Rating (1-6 Scale)',
        'Fleiss_Kappa': kappa,
        'Agreement_Level': kappa_level(kappa) if not np.isnan(kappa) else 'No data',
        'N_Questions': int(len(ratings_matrix)),
        'N_Raters': int(pivot.shape[1]) if pivot.shape[0] > 0 else 0,
        'Mean_Rating': round(mean_rating, 2) if not np.isnan(mean_rating) else np.nan,
        'Std_Rating': round(std_rating, 2) if not np.isnan(std_rating) else np.nan
    })

    return pd.DataFrame(results)

In [ ]:
kappa_all_df = calculate_fleiss(df_numeric, metrics)
print("\nFleiss' Kappa Calculation:")
display(kappa_all_df.round(4))
if 'output_tables_path' in locals():
    kappa_all_df.to_csv(os.path.join(output_tables_path, "exp2_fleiss_kappa_all_categories.csv"), index=False)
    print(f"Saved Fleiss' Kappa (all categories) table to {output_tables_path}")

In [ ]:
def save_results():
    print("Saving key results (based on all ratings)...")
    
    if 'overall_stats' in locals() and not overall_stats.empty:
        overall_stats.to_csv(os.path.join(output_tables_path, "exp2_overall_statistics_all.csv"))
        print("Saved overall statistics (all ratings)")
    
    if 'exp_stats' in locals() and not exp_stats.empty:
        exp_stats.to_csv(os.path.join(output_tables_path, "exp2_experiment_statistics_all.csv"))
        print("Saved experiment statistics (all ratings)")
    
    if 'kappa_all_df' in locals() and not kappa_all_df.empty:
        kappa_all_df.to_csv(os.path.join(output_tables_path, "exp2_fleiss_kappa_all.csv"), index=False)
        print("Saved Fleiss' Kappa results (all ratings)")
    
    print(f"\nResults saved to: {output_tables_path}")

save_results()
print(f"\n" + "="*60)
print("ANALYSIS COMPLETE")